## About

Few parts of this book is based on,

* https://www.kaggle.com/namanj27/new-baseline-pytorch-moa
* https://www.kaggle.com/felipebihaiek/prediction-with-swap-auto-encoder-features-0-01865
* https://www.kaggle.com/frtgnn/introduction-to-pytorch-a-very-gentle-start

In [1]:
!pip install -r /kaggle/input/python-library/wheelhouse/requirements.txt --no-index --find-links /kaggle/input/python-library/wheelhouse

Looking in links: /kaggle/input/python-library/wheelhouse
Processing /kaggle/input/python-library/wheelhouse/torch_optimizer-0.0.1a17-py3-none-any.whl
Processing /kaggle/input/python-library/wheelhouse/lambda_networks-0.4.0-py3-none-any.whl
Processing /kaggle/input/python-library/wheelhouse/torchsummary-1.5.1-py3-none-any.whl
Processing /kaggle/input/python-library/wheelhouse/iterative_stratification-0.1.6-py3-none-any.whl
Processing /kaggle/input/python-library/wheelhouse/pytorch_tabnet-2.0.1-py3-none-any.whl
Processing /kaggle/input/python-library/wheelhouse/pytorch_ranger-0.1.1-py3-none-any.whl
Processing /kaggle/input/python-library/wheelhouse/einops-0.3.0-py2.py3-none-any.whl


In [2]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch_optimizer
import lambda_networks
import torchsummary
import iterstrat

print(np.__version__)
print(pd.__version__)
print(torch.__version__)
print(torch_optimizer.__version__)
# print(lambda_networks.__version__)
# print(torchsummary.__version__)
print(iterstrat.__version__)

1.18.5
1.1.3
1.6.0
0.0.1a17
0.1.6


In [3]:
SEED = 42
SEEDS = [4, SEED, 6, 89]

def set_seed(seed):
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

N_EPOCHS = 250
N_FOLDS = 10
BATCH_SIZE = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Load dataset

In [4]:
X = pd.read_csv('/kaggle/input/lish-moa/train_features.csv')
y = pd.read_csv('/kaggle/input/lish-moa/train_targets_scored.csv')
X_test = pd.read_csv('/kaggle/input/lish-moa/test_features.csv')

In [5]:
df_submission = pd.DataFrame(columns=y.columns)
df_submission['sig_id'] = X_test['sig_id']

# Preprocess

## Preprocess - One Hot Encode

In [6]:
del X['sig_id']
del y['sig_id']
del X_test['sig_id']

In [7]:
cat_feature = ['cp_type', 'cp_dose']
for cat in cat_feature:
    val_list = X[cat].unique()
    for val in val_list:
        X[f'{cat}_{val}'] = X[cat].apply(lambda i:i == val)
        X_test[f'{cat}_{val}'] = X_test[cat].apply(lambda i:i == val)
        
    del X[cat]
    del X_test[cat]
    
print(X.columns[1:773])
print(X.columns[773:873])

Index(['g-0', 'g-1', 'g-2', 'g-3', 'g-4', 'g-5', 'g-6', 'g-7', 'g-8', 'g-9',
       ...
       'g-762', 'g-763', 'g-764', 'g-765', 'g-766', 'g-767', 'g-768', 'g-769',
       'g-770', 'g-771'],
      dtype='object', length=772)
Index(['c-0', 'c-1', 'c-2', 'c-3', 'c-4', 'c-5', 'c-6', 'c-7', 'c-8', 'c-9',
       'c-10', 'c-11', 'c-12', 'c-13', 'c-14', 'c-15', 'c-16', 'c-17', 'c-18',
       'c-19', 'c-20', 'c-21', 'c-22', 'c-23', 'c-24', 'c-25', 'c-26', 'c-27',
       'c-28', 'c-29', 'c-30', 'c-31', 'c-32', 'c-33', 'c-34', 'c-35', 'c-36',
       'c-37', 'c-38', 'c-39', 'c-40', 'c-41', 'c-42', 'c-43', 'c-44', 'c-45',
       'c-46', 'c-47', 'c-48', 'c-49', 'c-50', 'c-51', 'c-52', 'c-53', 'c-54',
       'c-55', 'c-56', 'c-57', 'c-58', 'c-59', 'c-60', 'c-61', 'c-62', 'c-63',
       'c-64', 'c-65', 'c-66', 'c-67', 'c-68', 'c-69', 'c-70', 'c-71', 'c-72',
       'c-73', 'c-74', 'c-75', 'c-76', 'c-77', 'c-78', 'c-79', 'c-80', 'c-81',
       'c-82', 'c-83', 'c-84', 'c-85', 'c-86', 'c-87', 'c-88', '

## Preprocess - Convert to Tensor

In [8]:
'''
1. Convert DataFrame data type to float32
2. Convert DataFrame to Numpy Array
3. Convert Numpy Array to Torch Tensor
'''

X = torch.tensor(X.astype(np.float32).to_numpy())
y = torch.tensor(y.astype(np.float32).to_numpy())
X_test = torch.tensor(X_test.astype(np.float32).to_numpy())

In [9]:
class MoADataset(torch.utils.data.Dataset):
    def __init__(self, X, y=None):
        if y is not None:
            assert X.shape[0] == y.shape[0]

        self.X = X
        self.y = y
        
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

In [10]:
train_ds = MoADataset(X, y)
test_ds = MoADataset(X_test)

# Autoencoder

## Autoencoder - Train function

In [11]:
def train_ae(mode, whole_dl):
    # Loss
    train_loss = 0
    lowest_loss = np.Inf
    
    last_epoch_with_lowest_loss = 0

    # Net
    if mode == 'AE_Gene':
        print('Train autoencoder gene!')
        net = AE_Gene_Net().float().to(device)
        criterion = nn.MSELoss()
        optimizer = torch_optimizer.RAdam(net.parameters(), lr=0.002, weight_decay=0.0000125)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer, factor=0.5, patience=8, min_lr = 0.00005, verbose=True
        )
    else:
        print('Train autoencoder cell!')
        net = AE_Cell_Net().float().to(device)
        criterion = nn.MSELoss()
        optimizer = torch_optimizer.RAdam(net.parameters(), lr=0.002, weight_decay=0.00001)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer, factor=0.25, patience=11, min_lr = 0.000025, verbose=True
        )

    # Train
    net.train()
    for epoch in range(N_EPOCHS):
        for X_local in whole_dl:
            X_local = X_local.to(device)

            optimizer.zero_grad()
            output = net(X_local)

            loss = criterion(output, X_local)          
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # post
        train_loss /= len(whole_dl)
        scheduler.step(train_loss)

        print(f'Epoch: {epoch+1:02d}/{N_EPOCHS:02d} | Train loss: {train_loss:.08f}{" (Saving model state!)" if train_loss < lowest_loss else ""}')
        if train_loss < lowest_loss:
            torch.save(net.state_dict(), f'./model_{mode}.state_dict')
            lowest_loss = train_loss
            last_epoch_with_lowest_loss = epoch
        if last_epoch_with_lowest_loss + 50 < epoch:
            print('Early stopping!')
            break

        train_loss = 0
    
    if mode == 'AE_Gene':
        net = AE_Gene_Net().float().to(device)
    else:
        net = AE_Cell_Net().float().to(device)
    net.load_state_dict(torch.load(f'./model_{mode}.state_dict'))
        
    return net, lowest_loss

## Autoencoder - Model

In [12]:
from torch import nn
from torch.nn import functional as F
from torchsummary import summary

class AE_Gene_Net(nn.Module):
    def __init__(self):
        super().__init__()
        
        # start
        self.encoder = nn.Sequential(
            nn.BatchNorm1d(772),
            nn.utils.weight_norm(nn.Linear(772, 541)),
            nn.ReLU6(),

            nn.BatchNorm1d(541),
            nn.utils.weight_norm(nn.Linear(541, 309)),
            nn.ReLU6(),
        )
        self.decoder = nn.Sequential(
            nn.BatchNorm1d(309),
            nn.utils.weight_norm(nn.Linear(309, 541)),
            nn.ReLU6(),

            nn.BatchNorm1d(541),
            nn.utils.weight_norm(nn.Linear(541, 772)),
        )

    def forward(self, x, mode='reconstruct'):
        x = self.encoder(x)
        if mode == 'reconstruct':
            x = self.decoder(x)
        
        return x
    
ae_gene_net = AE_Gene_Net().float().to(device)
print(ae_gene_net)
summary(ae_gene_net, (772, ))

AE_Gene_Net(
  (encoder): Sequential(
    (0): BatchNorm1d(772, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Linear(in_features=772, out_features=541, bias=True)
    (2): ReLU6()
    (3): BatchNorm1d(541, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Linear(in_features=541, out_features=309, bias=True)
    (5): ReLU6()
  )
  (decoder): Sequential(
    (0): BatchNorm1d(309, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Linear(in_features=309, out_features=541, bias=True)
    (2): ReLU6()
    (3): BatchNorm1d(541, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Linear(in_features=541, out_features=772, bias=True)
  )
)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
       BatchNorm1d-1                  [-1, 772]           1,544
            Linear-2                  [-1, 541]         418,193
           

In [13]:
class AE_Cell_Net(nn.Module):
    def __init__(self):
        super().__init__()
        
        # start
        self.encoder = nn.Sequential(
            nn.BatchNorm1d(100),
            nn.utils.weight_norm(nn.Linear(100, 70)),
            nn.ReLU6(),

            nn.BatchNorm1d(70),
            nn.utils.weight_norm(nn.Linear(70, 40)),
            nn.ReLU6(),
        )
        self.decoder = nn.Sequential(
            nn.BatchNorm1d(40),
            nn.utils.weight_norm(nn.Linear(40, 70)),
            nn.ReLU6(),

            nn.BatchNorm1d(70),
            nn.utils.weight_norm(nn.Linear(70, 100)),
        )

    def forward(self, x, mode='reconstruct'):
        x = self.encoder(x)
        if mode == 'reconstruct':
            x = self.decoder(x)
        
        return x
    
ae_cell_net = AE_Cell_Net().float().to(device)
print(ae_cell_net)
summary(ae_cell_net, (100, ))

AE_Cell_Net(
  (encoder): Sequential(
    (0): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Linear(in_features=100, out_features=70, bias=True)
    (2): ReLU6()
    (3): BatchNorm1d(70, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Linear(in_features=70, out_features=40, bias=True)
    (5): ReLU6()
  )
  (decoder): Sequential(
    (0): BatchNorm1d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Linear(in_features=40, out_features=70, bias=True)
    (2): ReLU6()
    (3): BatchNorm1d(70, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Linear(in_features=70, out_features=100, bias=True)
  )
)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
       BatchNorm1d-1                  [-1, 100]             200
            Linear-2                   [-1, 70]           7,070
             ReLU6-3

## Autoencoder - Train AE Gene

In [14]:
X_whole = torch.cat((X[:, 1:773], X_test[:, 1:773]), 0)

whole_ds = MoADataset(X_whole)
whole_dl = torch.utils.data.DataLoader(whole_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

ae_gene_net, ae_gene_min_loss = train_ae('AE_Gene', whole_dl)

Train autoencoder gene!
Epoch: 01/250 | Train loss: 0.90815236 (Saving model state!)
Epoch: 02/250 | Train loss: 0.61094128 (Saving model state!)
Epoch: 03/250 | Train loss: 0.51157396 (Saving model state!)
Epoch: 04/250 | Train loss: 0.41987572 (Saving model state!)
Epoch: 05/250 | Train loss: 0.36686242 (Saving model state!)
Epoch: 06/250 | Train loss: 0.33419977 (Saving model state!)
Epoch: 07/250 | Train loss: 0.31715483 (Saving model state!)
Epoch: 08/250 | Train loss: 0.29481872 (Saving model state!)
Epoch: 09/250 | Train loss: 0.28583724 (Saving model state!)
Epoch: 10/250 | Train loss: 0.28006078 (Saving model state!)
Epoch: 11/250 | Train loss: 0.27173386 (Saving model state!)
Epoch: 12/250 | Train loss: 0.26711153 (Saving model state!)
Epoch: 13/250 | Train loss: 0.27163160
Epoch: 14/250 | Train loss: 0.26143903 (Saving model state!)
Epoch: 15/250 | Train loss: 0.25527125 (Saving model state!)
Epoch: 16/250 | Train loss: 0.25368100 (Saving model state!)
Epoch: 17/250 | Train 

## Autoencoder - Train AE Cell

In [15]:
X_whole = torch.cat((X[:, 773:873], X_test[:, 773:873]), 0)

whole_ds = MoADataset(X_whole)
whole_dl = torch.utils.data.DataLoader(whole_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

ae_cell_net, ae_cell_min_loss = train_ae('AE_Cell', whole_dl)

Train autoencoder cell!
Epoch: 01/250 | Train loss: 1.76809768 (Saving model state!)
Epoch: 02/250 | Train loss: 0.60595172 (Saving model state!)
Epoch: 03/250 | Train loss: 0.50870973 (Saving model state!)
Epoch: 04/250 | Train loss: 0.46335392 (Saving model state!)
Epoch: 05/250 | Train loss: 0.43848112 (Saving model state!)
Epoch: 06/250 | Train loss: 0.40609160 (Saving model state!)
Epoch: 07/250 | Train loss: 0.39519845 (Saving model state!)
Epoch: 08/250 | Train loss: 0.38761976 (Saving model state!)
Epoch: 09/250 | Train loss: 0.36618786 (Saving model state!)
Epoch: 10/250 | Train loss: 0.36961317
Epoch: 11/250 | Train loss: 0.36431618 (Saving model state!)
Epoch: 12/250 | Train loss: 0.36065041 (Saving model state!)
Epoch: 13/250 | Train loss: 0.35797365 (Saving model state!)
Epoch: 14/250 | Train loss: 0.35122060 (Saving model state!)
Epoch: 15/250 | Train loss: 0.34308129 (Saving model state!)
Epoch: 16/250 | Train loss: 0.34111009 (Saving model state!)
Epoch: 17/250 | Train 

## Autoencoder - Transform Dataset

In [16]:
X_AE_Gene = torch.zeros([X.shape[0], 309])
X_AE_Cell = torch.zeros([X.shape[0], 40])

X_test_AE_Gene = torch.zeros([X_test.shape[0], 309])
X_test_AE_Cell = torch.zeros([X_test.shape[0], 40])

with torch.no_grad():
    for i in range(0, X.shape[0], BATCH_SIZE):
        start = i
        if start+BATCH_SIZE >= X.shape[0]:
            end = X.shape[0]
        else:
            end = start+BATCH_SIZE

        X_AE_Gene[start:end] = ae_gene_net(X[start:end, 1:773].to(device), mode='').cpu()
        X_AE_Cell[start:end] = ae_cell_net(X[start:end, 773:873].to(device), mode='').cpu()
    for i in range(0, X_test.shape[0], BATCH_SIZE):
        start = i
        if start+BATCH_SIZE >= X_test.shape[0]:
            end = X_test.shape[0]
        else:
            end = start+BATCH_SIZE

        X_test_AE_Gene[start:end] = ae_gene_net(X_test[start:end, 1:773].to(device), mode='').cpu()
        X_test_AE_Cell[start:end] = ae_cell_net(X_test[start:end, 773:873].to(device), mode='').cpu()

X = torch.cat((X, X_AE_Gene, X_AE_Cell), 1)
X_test = torch.cat((X_test, X_test_AE_Gene, X_test_AE_Cell), 1)

## Autoencoder - Feature Selection

In [17]:
from sklearn.feature_selection import VarianceThreshold

X_length, X_test_length = X.shape[0], X_test.shape[0]
X_temp = torch.cat((X, X_test), 0)
print('Shape before:', X_temp.shape)

selector = VarianceThreshold(0.5)
X_temp = selector.fit_transform(X_temp)
X_temp = torch.tensor(X_temp)
print('Shape after:', X_temp.shape)

X = X_temp[0:X_length]
X_test = X_temp[X_length:X_length+X_test_length]

Shape before: torch.Size([27796, 1226])
Shape after: torch.Size([27796, 863])


# MLP

## MLP - Train function

In [18]:
def train_mlp(train_dl, val_dl, seed, fold):
    # Loss
    train_loss = 0
    val_loss = 0
    lowest_val_loss = np.Inf
    
    last_epoch_with_lowest_val_loss = 0

    # Net
    net = MLPNet().float().to(device)
    criterion = nn.BCELoss()
#     optimizer = torch_optimizer.RAdam(net.parameters(), lr=0.001, weight_decay=0.00002)
#     scheduler = torch.optim.lr_scheduler.OneCycleLR(
#         optimizer=optimizer, pct_start=0.1, div_factor=1e3, 
#         max_lr=0.01, epochs=N_EPOCHS, steps_per_epoch=len(train_dl)
#     )
    optimizer = torch_optimizer.AdaBelief(net.parameters(), lr=0.1, weight_decay=0.00002, rectify=False, weight_decouple=False)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer, pct_start=0.1, div_factor=1e3, 
        max_lr=1.0, epochs=N_EPOCHS, steps_per_epoch=len(train_dl)
    )

    # Train/Eval
    for epoch in range(N_EPOCHS):
        # Train
        net.train()
        for X_local, y_local in train_dl:
            X_local = X_local.to(device)
            y_local = y_local.to(device)

            optimizer.zero_grad()
            output = net(X_local)

            loss = criterion(output, y_local)                
            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

        # Val
        net.eval()
        for X_local, y_local in val_dl:
            X_local = X_local.to(device)
            y_local = y_local.to(device)

            output = net(X_local)
            loss = criterion(output, y_local)   

            val_loss += loss.item()

        # post
        train_loss /= len(train_dl)
        val_loss /= len(val_dl)
#         scheduler.step(val_loss)

        print(f'Epoch: {epoch+1:02d}/{N_EPOCHS:02d} | Train loss: {train_loss:.08f} - Val loss: {val_loss:.08f}{" (Saving model state!)" if val_loss < lowest_val_loss else ""}')
        if val_loss < lowest_val_loss:
            torch.save(net.state_dict(), f'./model_mlp_{seed:02d}_{fold:02d}.state_dict')
            lowest_val_loss = val_loss
            last_epoch_with_lowest_val_loss = epoch
        if last_epoch_with_lowest_val_loss + 30 < epoch:
            print('Early stopping!')
            # stop training
            break

        train_loss = 0
        val_loss = 0

    net = MLPNet().float().to(device)
    net.load_state_dict(torch.load(f'./model_mlp_{seed:02d}_{fold:02d}.state_dict'))
        
    return net, lowest_val_loss

## MLP - Model

In [19]:
class MLPNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # start
        self.bn1 = nn.BatchNorm1d(X.shape[1])
        self.drop1 = nn.Dropout(0.2)
        self.fc1 = nn.utils.weight_norm(nn.Linear(X.shape[1], 1280))

        self.bn2 = nn.BatchNorm1d(1280)
        self.drop2 = nn.Dropout(0.5)
        self.fc2 = nn.utils.weight_norm(nn.Linear(1280, 1280))


        self.bn3 = nn.BatchNorm1d(1280)
        self.drop3 = nn.Dropout(0.5)
        self.fc3 = nn.utils.weight_norm(nn.Linear(1280, y.shape[1]))

        # generic
        self.relu = nn.ReLU6()
        self.sigmoid = nn.Sigmoid()
        

    def forward(self, x):
        x = self.bn1(x)
        x = self.drop1(x)
        x = self.fc1(x)
        x = self.relu(x)

        x = self.bn2(x)
        x = self.drop2(x)
        x = self.fc2(x)
        x = self.relu(x)

        x = self.bn3(x)
        x = self.drop3(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        
        return x
    
mlp_net = MLPNet().float().to(device)
print(mlp_net)
summary(mlp_net, (X.shape[1], ))

MLPNet(
  (bn1): BatchNorm1d(863, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=863, out_features=1280, bias=True)
  (bn2): BatchNorm1d(1280, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop2): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=1280, out_features=1280, bias=True)
  (bn3): BatchNorm1d(1280, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop3): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=1280, out_features=206, bias=True)
  (relu): ReLU6()
  (sigmoid): Sigmoid()
)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
       BatchNorm1d-1                  [-1, 863]           1,726
           Dropout-2                  [-1, 863]               0
            Linear-3                 [-1, 1280]       1,105,920
             ReLU6-4                 [-1, 1

## MLP - Train

In [20]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

mlp_nets = []
mlp_min_losses = []

for seed in SEEDS:
    set_seed(seed)
    skf = MultilabelStratifiedKFold(n_splits=N_FOLDS, random_state=seed, shuffle=True)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f'Seed: {seed}, fold: {fold+1}/{N_FOLDS}')

        X_train, y_train = X[train_idx], y[train_idx]
        train_ds = MoADataset(X_train, y_train)
        train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

        X_val, y_val = X[val_idx], y[val_idx]
        val_ds = MoADataset(X_val, y_val)
        val_dl = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        mlp_net, mlp_min_loss = train_mlp(train_dl, val_dl, seed, fold)
        mlp_nets.append(mlp_net)
        mlp_min_losses.append(mlp_min_loss)
                
        print('='*82)
print('Training Ended! ')

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass shuffle=True, random_state=4 as keyword args. From version 0.25 passing these as positional arguments will result in an error
  FutureWarning)


Seed: 4, fold: 1/10
Epoch: 01/250 | Train loss: 0.76712862 - Val loss: 0.70947636 (Saving model state!)
Epoch: 02/250 | Train loss: 0.76460623 - Val loss: 0.70554514 (Saving model state!)
Epoch: 03/250 | Train loss: 0.75683922 - Val loss: 0.69571849 (Saving model state!)
Epoch: 04/250 | Train loss: 0.74251497 - Val loss: 0.67915702 (Saving model state!)
Epoch: 05/250 | Train loss: 0.71773298 - Val loss: 0.65412592 (Saving model state!)
Epoch: 06/250 | Train loss: 0.67449372 - Val loss: 0.59989673 (Saving model state!)
Epoch: 07/250 | Train loss: 0.58068131 - Val loss: 0.46070440 (Saving model state!)
Epoch: 08/250 | Train loss: 0.36270907 - Val loss: 0.19552846 (Saving model state!)
Epoch: 09/250 | Train loss: 0.12967762 - Val loss: 0.06297825 (Saving model state!)
Epoch: 10/250 | Train loss: 0.05213481 - Val loss: 0.03410629 (Saving model state!)
Epoch: 11/250 | Train loss: 0.03267286 - Val loss: 0.02553816 (Saving model state!)
Epoch: 12/250 | Train loss: 0.02587707 - Val loss: 0.022

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass shuffle=True, random_state=42 as keyword args. From version 0.25 passing these as positional arguments will result in an error
  FutureWarning)


Seed: 42, fold: 1/10
Epoch: 01/250 | Train loss: 0.76670730 - Val loss: 0.71004066 (Saving model state!)
Epoch: 02/250 | Train loss: 0.76396763 - Val loss: 0.70535104 (Saving model state!)
Epoch: 03/250 | Train loss: 0.75661483 - Val loss: 0.69542599 (Saving model state!)
Epoch: 04/250 | Train loss: 0.74179104 - Val loss: 0.68084749 (Saving model state!)
Epoch: 05/250 | Train loss: 0.71747695 - Val loss: 0.65376355 (Saving model state!)
Epoch: 06/250 | Train loss: 0.67300607 - Val loss: 0.59960441 (Saving model state!)
Epoch: 07/250 | Train loss: 0.57704829 - Val loss: 0.45721839 (Saving model state!)
Epoch: 08/250 | Train loss: 0.35407844 - Val loss: 0.19252451 (Saving model state!)
Epoch: 09/250 | Train loss: 0.12652457 - Val loss: 0.06383004 (Saving model state!)
Epoch: 10/250 | Train loss: 0.05129130 - Val loss: 0.03453127 (Saving model state!)
Epoch: 11/250 | Train loss: 0.03242492 - Val loss: 0.02585663 (Saving model state!)
Epoch: 12/250 | Train loss: 0.02595933 - Val loss: 0.02

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass shuffle=True, random_state=6 as keyword args. From version 0.25 passing these as positional arguments will result in an error
  FutureWarning)


Seed: 6, fold: 1/10
Epoch: 01/250 | Train loss: 0.76608157 - Val loss: 0.70570103 (Saving model state!)
Epoch: 02/250 | Train loss: 0.76363948 - Val loss: 0.70065862 (Saving model state!)
Epoch: 03/250 | Train loss: 0.75611247 - Val loss: 0.69119866 (Saving model state!)
Epoch: 04/250 | Train loss: 0.74146407 - Val loss: 0.67575020 (Saving model state!)
Epoch: 05/250 | Train loss: 0.71670320 - Val loss: 0.64934110 (Saving model state!)
Epoch: 06/250 | Train loss: 0.67319489 - Val loss: 0.59500600 (Saving model state!)
Epoch: 07/250 | Train loss: 0.57638163 - Val loss: 0.45299675 (Saving model state!)
Epoch: 08/250 | Train loss: 0.35568446 - Val loss: 0.18891687 (Saving model state!)
Epoch: 09/250 | Train loss: 0.12717932 - Val loss: 0.06211233 (Saving model state!)
Epoch: 10/250 | Train loss: 0.05124300 - Val loss: 0.03378461 (Saving model state!)
Epoch: 11/250 | Train loss: 0.03254442 - Val loss: 0.02567619 (Saving model state!)
Epoch: 12/250 | Train loss: 0.02588722 - Val loss: 0.022

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass shuffle=True, random_state=89 as keyword args. From version 0.25 passing these as positional arguments will result in an error
  FutureWarning)


Seed: 89, fold: 1/10
Epoch: 01/250 | Train loss: 0.76564679 - Val loss: 0.72171530 (Saving model state!)
Epoch: 02/250 | Train loss: 0.76299139 - Val loss: 0.71527049 (Saving model state!)
Epoch: 03/250 | Train loss: 0.75537509 - Val loss: 0.70420098 (Saving model state!)
Epoch: 04/250 | Train loss: 0.74088452 - Val loss: 0.68703543 (Saving model state!)
Epoch: 05/250 | Train loss: 0.71625673 - Val loss: 0.66193942 (Saving model state!)
Epoch: 06/250 | Train loss: 0.67289819 - Val loss: 0.61132817 (Saving model state!)
Epoch: 07/250 | Train loss: 0.57798504 - Val loss: 0.46979950 (Saving model state!)
Epoch: 08/250 | Train loss: 0.35921847 - Val loss: 0.20247859 (Saving model state!)
Epoch: 09/250 | Train loss: 0.12933931 - Val loss: 0.06698983 (Saving model state!)
Epoch: 10/250 | Train loss: 0.05160649 - Val loss: 0.03503775 (Saving model state!)
Epoch: 11/250 | Train loss: 0.03247274 - Val loss: 0.02627649 (Saving model state!)
Epoch: 12/250 | Train loss: 0.02599926 - Val loss: 0.02

# Post-training

In [21]:
print(f'AE Gene loss: {ae_gene_min_loss}')
print(f'AE Cell loss: {ae_cell_min_loss}')

print('MLP CV min loss:')
[print(l) for l in mlp_min_losses]

for i in range(len(SEEDS)):
    print(f'MLP average CV min loss (SEED {SEEDS[i]}): {sum(mlp_min_losses[i*N_FOLDS:i*N_FOLDS+N_FOLDS])/N_FOLDS}')
print(f'MLP average CV min loss: {sum(mlp_min_losses)/len(mlp_min_losses)}')

AE Gene loss: 0.20242725066635586
AE Cell loss: 0.24207444338623538
MLP CV min loss:
0.014867670196843775
0.015030479744861
0.015059101199241061
0.015175162275370798
0.014904692621999666
0.015068787787305681
0.015248199042521025
0.01526898827011648
0.015090286290567172
0.014928841277172691
0.015107786380930951
0.014930594663478826
0.01493170249618982
0.015226971112975949
0.014947866579811824
0.014780591977270026
0.01506281359807441
0.015045917739993647
0.015234161217353846
0.015368442443248472
0.015012379343572416
0.015083660469635538
0.015133530195606383
0.01526229279605966
0.014977344605875643
0.014964140657531587
0.01500215028461657
0.01509791264604581
0.015064499438985399
0.015097161362829962
0.014743304272231302
0.015244720699755769
0.015370009025852931
0.01512429858312795
0.015033421106636524
0.015292098794720675
0.015108917153587467
0.01497392160327811
0.015028270990832857
0.014916030494006057
MLP average CV min loss (SEED 4): 0.015064220870599937
MLP average CV min loss (SEED 4

In [22]:
with torch.no_grad():
    for i in range(0, X_test.shape[0], BATCH_SIZE):
        start = i
        if start+BATCH_SIZE >= X_test.shape[0]:
            end = X_test.shape[0]
        else:
            end = start+BATCH_SIZE

        test_results = torch.zeros([end-start, y.shape[1]])
        for mlp_net in mlp_nets:
            test_results += mlp_net(X_test[start:end].to(device)).cpu()
        test_results /= int(N_FOLDS * len(SEEDS))

        df_submission.iloc[start:end, 1:] = test_results

In [23]:
df_submission

,sig_id,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
0,id_0004d9e33,0.00104549,0.00181359,0.00188289,0.0208927,0.0202598,0.00422637,0.00163297,0.00577574,0.000304308,...,0.00123913,0.00384225,0.00480124,0.00175708,0.000550864,0.000830022,0.00167308,0.00190061,0.00775239,0.00199342
1,id_001897cda,0.000284258,0.00111217,0.00169029,0.000882401,0.00151199,0.00262166,0.00257787,0.00686282,0.00661273,...,0.000569681,0.000983269,0.00258279,0.000361604,0.00677765,0.000529504,0.00418174,0.000839191,0.00402293,0.00285569
2,id_002429b5b,0.000265425,0.000167538,0.00151787,0.00554841,0.01115,0.000826827,0.00175551,0.00132621,0.000254491,...,0.000571476,0.000526154,0.0018234,0.011848,0.00214246,0.000199716,0.00232074,0.00225066,0.000849513,0.00133324
3,id_00276f245,0.00036358,0.000436513,0.00140698,0.00609942,0.00411979,0.00298147,0.0017299,0.00265984,0.000304371,...,0.000520224,0.000798144,0.00359668,0.0172459,0.00366921,0.000355483,0.00195119,0.0013271,0.00103233,0.00220088
4,id_0027f1083,0.00158923,0.00138361,0.00244608,0.0146087,0.0144319,0.00236246,0.00567499,0.00189077,0.000575292,...,0.000920899,0.000661495,0.00586261,0.00610961,0.000841933,0.00078461,0.00232705,0.00187328,0.000469759,0.00157459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3977,id_ff7004b87,0.000598491,0.0011951,0.00148447,0.00301266,0.0118826,0.00180001,0.00128743,0.00315623,0.000867629,...,0.000610975,0.00330577,0.00184986,0.0975304,0.00740435,0.000765361,0.00557478,0.00120654,0.00146807,0.0013918
3978,id_ff925dd0d,0.00448801,0.00212367,0.000677733,0.00480061,0.0129375,0.00510045,0.00568558,0.00186449,0.00099737,...,0.000393625,0.000883801,0.00151402,0.000452565,0.00116455,0.000669109,0.000444229,0.000834292,0.000909868,0.0010612
3979,id_ffb710450,0.00173965,0.00101314,0.000959656,0.0116371,0.0539304,0.00588207,0.00403223,0.00375766,0.000402593,...,0.000492698,0.000405489,0.00175009,0.000439137,0.00135095,0.000541522,0.000461727,0.00103236,0.000720569,0.00148751
3980,id_ffbb869f2,0.000753423,0.000927147,0.00107093,0.0236548,0.0123692,0.00358569,0.00344107,0.00172318,0.000616288,...,0.000370892,0.000326803,0.00277343,0.00118703,0.00106103,0.000354374,0.00490244,0.00115253,0.000517133,0.00255433


In [24]:
df_submission.to_csv('submission.csv', index=False)